# Tutorial 03 - Project 2: Student Cohort Simulation & Analytics

Generates synthetic student data with random birth dates (2000-2006) and
grades, computes descriptive statistics, and exports an executive report.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/03_Student_Simulation_Analytics/` project folder (anchored on `students.py` — the modules `students` and `analysis` import from it) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "03_Student_Simulation_Analytics"
ANCHOR_REL = "students.py"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Generate and Enrich the Student Cohort


In [ ]:
from students import generate_students
from analysis import enrich_student_data, compute_cohort_statistics
import os

raw_cohort = generate_students(num_students=10, seed=101)
cohort = enrich_student_data(raw_cohort, reference_year=2026)
stats = compute_cohort_statistics(cohort)

print("--- Simulated Student Records ---")
for s in cohort:
    print(f"ID: {s['student_id']} | Age: {s['age']} | Grades: {s['grades']} | Average: {s['average']}")


### 2. Summary Statistics Overview


In [ ]:
for k, v in stats.items():
    print(f"{k.replace('_', ' ').capitalize():15s}: {v}")


### 3. Export the Audit Report
The report is written to the project's `results/report.txt` (resolved from
the project folder, not the working directory).


In [ ]:
report_path = PROJ_DIR / "results" / "report.txt"
os.makedirs(PROJ_DIR / "results", exist_ok=True)

with open(report_path, "w", encoding="utf-8") as f:
    f.write("==============================================\n")
    f.write("     COHORT ACADEMIC STATISTICAL REPORT       \n")
    f.write("==============================================\n")
    for k, v in stats.items():
        f.write(f"{k.replace('_', ' ').capitalize():15s}: {v}\n")
    f.write("----------------------------------------------\n")
    for s in cohort:
        f.write(f"{s['student_id']} | Age: {s['age']} | Avg: {s['average']}\n")

print(f"Report written to: {report_path}")
